# Importing

In [2]:
import pandas as pd
import os
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

sim_data_path = '../SimulationData/'
sim_image_path = '../SimulationData/figures/'

meta_df = pd.read_csv(sim_data_path + "simulation_metadata.csv")

# Plotting

In [3]:
def convert_id_to_group_label(id_string):
    """
    Convert dataset IDs like 'piecewise_low_low_low_0_0' to 'LLL'.
    """
    tokens = id_string.split('_')
    level_tokens = []

    for token in tokens[1:]:
        if token.isdigit():
            break
        level_tokens.append(token)

    mapping = {
        'low': 'L',
        'medium': 'M',
        'high': 'H'
    }

    group_label = ''.join(mapping.get(token.lower(), token) for token in level_tokens)
    return group_label

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from scipy import interpolate
import warnings
warnings.filterwarnings('ignore')

def calculate_km_survival_quantiles(df, quantiles=[0.975, 0.5, 0.025]):
    """
    Calculate Kaplan-Meier survival probabilities at given quantiles.
    
    Parameters:
    df (pd.DataFrame): DataFrame with 'Time' and 'Event' columns
    quantiles (list): List of survival probability quantiles to evaluate
    
    Returns:
    dict: Dictionary with quantiles as keys and corresponding times as values
    """
    kmf = KaplanMeierFitter()
    kmf.fit(df['Time'], df['Event'])
    
    # Get survival function
    survival_function = kmf.survival_function_
    times = survival_function.index
    probabilities = survival_function.iloc[:, 0].values
    
    # Calculate quantiles
    quantile_times = {}
    
    for q in quantiles:
        # Find the time where survival probability crosses the quantile
        # Use interpolation for more accurate estimates
        if q <= probabilities.min():
            # Quantile is below minimum survival probability
            quantile_times[q] = np.inf
        elif q >= probabilities.max():
            # Quantile is above maximum survival probability
            quantile_times[q] = times.min()
        else:
            # Interpolate to find exact crossing point
            # Reverse arrays since survival function is decreasing
            f_interp = interpolate.interp1d(probabilities[::-1], times[::-1], 
                                          kind='linear', bounds_error=False, 
                                          fill_value='extrapolate')
            quantile_times[q] = f_interp(q)
    
    return quantile_times, kmf

def calculate_survival_mae_landmarks(truth_df, reconstructed_df, percentiles=[0.95, 0.5, 0.025]):
    """
    Calculate Mean Absolute Error between survival probabilities at landmark times.
    Landmark times are defined as percentiles of the maximum study time.
    
    Parameters:
    truth_df (pd.DataFrame): Ground truth DataFrame with 'Time' and 'Event' columns
    reconstructed_df (pd.DataFrame): Reconstructed DataFrame with 'Time' and 'Event' columns
    percentiles (list): List of percentiles to use as landmark times (e.g., [0.95, 0.5, 0.025])
    
    Returns:
    pd.DataFrame: DataFrame with columns ['percentile', 'landmark_time', 'truth_survival_prob', 
                  'reconstructed_survival_prob', 'absolute_error', 'mae']
    """
    # Find maximum study time across both datasets
    max_study_time = max(truth_df['Time'].max(), reconstructed_df['Time'].max())
    
    # Calculate landmark times as percentiles of max study time
    landmark_times = [p * max_study_time for p in percentiles]
    
    # Fit Kaplan-Meier estimators
    truth_kmf = KaplanMeierFitter()
    truth_kmf.fit(truth_df['Time'], truth_df['Event'])
    
    recon_kmf = KaplanMeierFitter()
    recon_kmf.fit(reconstructed_df['Time'], reconstructed_df['Event'])
    
    # Prepare data for DataFrame
    results_data = []
    absolute_errors = []
    
    for i, percentile in enumerate(percentiles):
        landmark_time = landmark_times[i]
        
        # Get survival probabilities at landmark time
        # Use predict_survival_function to get probability at specific time
        truth_survival_prob = truth_kmf.survival_function_at_times(landmark_time).iloc[0]
        recon_survival_prob = recon_kmf.survival_function_at_times(landmark_time).iloc[0]
        
        # Calculate absolute error
        absolute_error = abs(truth_survival_prob - recon_survival_prob)
        absolute_errors.append(absolute_error)
        
        # Append row data
        results_data.append({
            'percentile': percentile,
            'landmark_time': landmark_time,
            'truth_survival_prob': truth_survival_prob,
            'reconstructed_survival_prob': recon_survival_prob,
            'absolute_error': absolute_error
        })
    
    # Create DataFrame
    results_df = pd.DataFrame(results_data)
    
    # Store additional information as attributes
    results_df.attrs['truth_kmf'] = truth_kmf
    results_df.attrs['reconstructed_kmf'] = recon_kmf
    results_df.attrs['max_study_time'] = max_study_time
    results_df.attrs['landmark_times'] = landmark_times
    
    return results_df

def integrated_abs_error_km(truth_df: pd.DataFrame, reconstruct_df: pd.DataFrame) -> float:
    """
    Compute the integrated absolute error between two KM survival curves with time normalized to [0,1].
    """
    # Work on copies
    tdf = truth_df[['Time', 'Event']].copy()
    rdf = reconstruct_df[['Time', 'Event']].copy()

    # Normalize time to [0,1]
    # Guard against zero max to avoid division by zero
    t_max = max(float(tdf['Time'].max()), 1e-12)
    r_max = max(float(rdf['Time'].max()), 1e-12)
    tdf['t_norm'] = tdf['Time'] / t_max
    rdf['t_norm'] = rdf['Time'] / r_max

    # Fit KM on normalized times
    kmf_true = KaplanMeierFitter()
    kmf_reco = KaplanMeierFitter()
    kmf_true.fit(durations=tdf['t_norm'], event_observed=tdf['Event'])
    kmf_reco.fit(durations=rdf['t_norm'], event_observed=rdf['Event'])

    # Build a common grid from both sets of event times, include endpoints [0,1]
    grid = np.unique(
        np.clip(
            np.concatenate([
                np.array([0.0, 1]),
                tdf['t_norm'].to_numpy(),
                rdf['t_norm'].to_numpy()
            ]),
            0.0, 1
        )
    )
    grid.sort()

    # Stepwise predictions (no interpolation between steps)
    # lifelines KM predict defaults to previous-step value when interpolate=False
    S_true = kmf_true.predict(grid, interpolate=False).to_numpy()
    S_reco = kmf_reco.predict(grid, interpolate=False).to_numpy()

    # Integrated absolute error over [0,1]
    abs_diff = np.abs(S_true - S_reco)
    if hasattr(np, 'trapezoid'):
        iae = np.trapezoid(abs_diff, grid)
    else:
        iae = np.trapz(abs_diff, grid)
    return float(iae)

# GPT-5 Minimal

In [5]:
meta_df = pd.read_csv(sim_data_path + 'simulation_metadata.csv')

failed_list = []
records = []  # we'll collect rows and build a DataFrame once (faster than repeated concat)

for index, row in tqdm(meta_df.iterrows(), total=len(meta_df)):
    unique_id = row['dataset_id']
    group_label = convert_id_to_group_label(unique_id)

    try:
        truth_df = pd.read_csv(sim_data_path + f'ipd/simulated_ipd_{unique_id}.csv')
        reconstruct_df = pd.read_csv(sim_data_path + f"recon_GPT5_minimal/ipd/{unique_id}_recon_ipd.csv")
        reconstruct_df = reconstruct_df.rename(columns={'time': 'Time', 'status': 'Event'})
    except Exception as e:
        # print(e)
        failed_list.append(unique_id)
        continue

    try:
        iae = integrated_abs_error_km(truth_df, reconstruct_df)
        records.append({
            'dataset_id': unique_id,
            'group': group_label,
            'integrated_abs_error': iae
        })
    except Exception as e:
        # print(e)
        failed_list.append(unique_id)
        continue

final_result_df = pd.DataFrame.from_records(records)

group_order = final_result_df['group'].unique()

print(len(failed_list), "datasets failed to process")
print(1-len(failed_list)/len(meta_df), "success rate")

q_values = final_result_df['integrated_abs_error'].quantile([0.025, 0.5, 0.975])
print(f"IAE: {q_values.loc[0.5]:.4f} [{q_values.loc[0.025]:.4f}, {q_values.loc[0.975]:.4f}]")

  0%|          | 0/1620 [00:00<?, ?it/s]

100%|██████████| 1620/1620 [00:47<00:00, 34.29it/s]

0 datasets failed to process
1.0 success rate
IAE: 0.0067 [0.0018, 0.0397]


# Qwen32B

## T = 1

In [6]:
meta_df = pd.read_csv(sim_data_path + 'simulation_metadata.csv')

failed_list = []
records = []  # we'll collect rows and build a DataFrame once (faster than repeated concat)

for index, row in tqdm(meta_df.iterrows(), total=len(meta_df)):
    unique_id = row['dataset_id']
    group_label = convert_id_to_group_label(unique_id)

    try:
        truth_df = pd.read_csv(sim_data_path + f'ipd/simulated_ipd_{unique_id}.csv')
        reconstruct_df = pd.read_csv(sim_data_path + f"recon_Qwen32B_Tat1/ipd/{unique_id}_recon_ipd.csv")
        reconstruct_df = reconstruct_df.rename(columns={'time': 'Time', 'status': 'Event'})
    except Exception as e:
        # print(e)
        failed_list.append(unique_id)
        continue

    try:
        iae = integrated_abs_error_km(truth_df, reconstruct_df)
        records.append({
            'dataset_id': unique_id,
            'group': group_label,
            'integrated_abs_error': iae
        })
    except Exception as e:
        # print(e)
        failed_list.append(unique_id)
        continue

final_result_df = pd.DataFrame.from_records(records)

group_order = final_result_df['group'].unique()

print(len(failed_list), "datasets failed to process")
print(1-len(failed_list)/len(meta_df), "success rate")

q_values = final_result_df['integrated_abs_error'].quantile([0.025, 0.5, 0.975])
print(f"IAE: {q_values.loc[0.5]:.4f} [{q_values.loc[0.025]:.4f}, {q_values.loc[0.975]:.4f}]")

100%|██████████| 1620/1620 [00:36<00:00, 44.15it/s]

1 datasets failed to process
0.9993827160493827 success rate
IAE: 0.0132 [0.0021, 0.1307]


## T = 0

In [7]:
meta_df = pd.read_csv(sim_data_path + 'simulation_metadata.csv')

failed_list = []
records = []  # we'll collect rows and build a DataFrame once (faster than repeated concat)

for index, row in tqdm(meta_df.iterrows(), total=len(meta_df)):
    unique_id = row['dataset_id']
    group_label = convert_id_to_group_label(unique_id)

    try:
        truth_df = pd.read_csv(sim_data_path + f'ipd/simulated_ipd_{unique_id}.csv')
        reconstruct_df = pd.read_csv(sim_data_path + f"recon_Qwen32B_Tat0/ipd/{unique_id}_recon_ipd.csv")
        reconstruct_df = reconstruct_df.rename(columns={'time': 'Time', 'status': 'Event'})
    except Exception as e:
        # print(e)
        failed_list.append(unique_id)
        continue

    try:
        iae = integrated_abs_error_km(truth_df, reconstruct_df)
        records.append({
            'dataset_id': unique_id,
            'group': group_label,
            'integrated_abs_error': iae
        })
    except Exception as e:
        # print(e)
        failed_list.append(unique_id)
        continue

final_result_df = pd.DataFrame.from_records(records)

group_order = final_result_df['group'].unique()

print(len(failed_list), "datasets failed to process")
print(1-len(failed_list)/len(meta_df), "success rate")

q_values = final_result_df['integrated_abs_error'].quantile([0.025, 0.5, 0.975])
print(f"IAE: {q_values.loc[0.5]:.4f} [{q_values.loc[0.025]:.4f}, {q_values.loc[0.975]:.4f}]")

100%|██████████| 1620/1620 [00:38<00:00, 41.58it/s]

6 datasets failed to process
0.9962962962962963 success rate
IAE: 0.0072 [0.0018, 0.0587]


# GPT 4o

## T = 1

In [8]:
meta_df = pd.read_csv(sim_data_path + 'simulation_metadata.csv')

failed_list = []
records = []  # we'll collect rows and build a DataFrame once (faster than repeated concat)

for index, row in tqdm(meta_df.iterrows(), total=len(meta_df)):
    unique_id = row['dataset_id']
    group_label = convert_id_to_group_label(unique_id)

    try:
        truth_df = pd.read_csv(sim_data_path + f'ipd/simulated_ipd_{unique_id}.csv')
        reconstruct_df = pd.read_csv(sim_data_path + f"recon_GPT4o_Tat1/ipd/{unique_id}_recon_ipd.csv")
        reconstruct_df = reconstruct_df.rename(columns={'time': 'Time', 'status': 'Event'})
    except Exception as e:
        # print(e)
        failed_list.append(unique_id)
        continue

    try:
        iae = integrated_abs_error_km(truth_df, reconstruct_df)
        records.append({
            'dataset_id': unique_id,
            'group': group_label,
            'integrated_abs_error': iae
        })
    except Exception as e:
        # print(e)
        failed_list.append(unique_id)
        continue

final_result_df = pd.DataFrame.from_records(records)

group_order = final_result_df['group'].unique()

print(len(failed_list), "datasets failed to process")
print(1-len(failed_list)/len(meta_df), "success rate")

q_values = final_result_df['integrated_abs_error'].quantile([0.025, 0.5, 0.975])
print(f"IAE: {q_values.loc[0.5]:.4f} [{q_values.loc[0.025]:.4f}, {q_values.loc[0.975]:.4f}]")

100%|██████████| 1620/1620 [00:29<00:00, 54.59it/s] 

23 datasets failed to process
0.9858024691358025 success rate
IAE: 0.0137 [0.0023, 0.1342]


## T = 0.5

In [9]:
meta_df = pd.read_csv(sim_data_path + 'simulation_metadata.csv')

failed_list = []
records = []  # we'll collect rows and build a DataFrame once (faster than repeated concat)

for index, row in tqdm(meta_df.iterrows(), total=len(meta_df)):
    unique_id = row['dataset_id']
    group_label = convert_id_to_group_label(unique_id)

    try:
        truth_df = pd.read_csv(sim_data_path + f'ipd/simulated_ipd_{unique_id}.csv')
        reconstruct_df = pd.read_csv(sim_data_path + f"recon_GPT4o_Tat05/ipd/{unique_id}_recon_ipd.csv")
        reconstruct_df = reconstruct_df.rename(columns={'time': 'Time', 'status': 'Event'})
    except Exception as e:
        # print(e)
        failed_list.append(unique_id)
        continue

    try:
        iae = integrated_abs_error_km(truth_df, reconstruct_df)
        records.append({
            'dataset_id': unique_id,
            'group': group_label,
            'integrated_abs_error': iae
        })
    except Exception as e:
        # print(e)
        failed_list.append(unique_id)
        continue

final_result_df = pd.DataFrame.from_records(records)

group_order = final_result_df['group'].unique()

print(len(failed_list), "datasets failed to process")
print(1-len(failed_list)/len(meta_df), "success rate")

q_values = final_result_df['integrated_abs_error'].quantile([0.025, 0.5, 0.975])
print(f"IAE: {q_values.loc[0.5]:.4f} [{q_values.loc[0.025]:.4f}, {q_values.loc[0.975]:.4f}]")

100%|██████████| 1620/1620 [00:28<00:00, 56.77it/s]

3 datasets failed to process
0.9981481481481481 success rate
IAE: 0.0139 [0.0023, 0.1196]


## T = 0

In [10]:
meta_df = pd.read_csv(sim_data_path + 'simulation_metadata.csv')

failed_list = []
records = []  # we'll collect rows and build a DataFrame once (faster than repeated concat)

for index, row in tqdm(meta_df.iterrows(), total=len(meta_df)):
    unique_id = row['dataset_id']
    group_label = convert_id_to_group_label(unique_id)

    try:
        truth_df = pd.read_csv(sim_data_path + f'ipd/simulated_ipd_{unique_id}.csv')
        reconstruct_df = pd.read_csv(sim_data_path + f"recon_GPT4o_Tat0/ipd/{unique_id}_recon_ipd.csv")
        reconstruct_df = reconstruct_df.rename(columns={'time': 'Time', 'status': 'Event'})
    except Exception as e:
        # print(e)
        failed_list.append(unique_id)
        continue

    try:
        iae = integrated_abs_error_km(truth_df, reconstruct_df)
        records.append({
            'dataset_id': unique_id,
            'group': group_label,
            'integrated_abs_error': iae
        })
    except Exception as e:
        # print(e)
        failed_list.append(unique_id)
        continue

final_result_df = pd.DataFrame.from_records(records)

group_order = final_result_df['group'].unique()

print(len(failed_list), "datasets failed to process")
print(1-len(failed_list)/len(meta_df), "success rate")

q_values = final_result_df['integrated_abs_error'].quantile([0.025, 0.5, 0.975])
print(f"IAE: {q_values.loc[0.5]:.4f} [{q_values.loc[0.025]:.4f}, {q_values.loc[0.975]:.4f}]")

100%|██████████| 1620/1620 [00:30<00:00, 52.28it/s]

1 datasets failed to process
0.9993827160493827 success rate
IAE: 0.0140 [0.0023, 0.1229]
